In [ ]:
# Setting system path and project root
import os
import sys
from datetime import datetime

PROJECT_ROOT_DIR = os.path.abspath('../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

def get_fp(relative_path):
    return os.path.join(PROJECT_ROOT_DIR, relative_path)

In [ ]:
import src.const.misc as misc_consts
from src.util.common import create_directory_if_not_exists
# init sparql logger
now = datetime.now()
timestamp = now.strftime("%Y-%m-%d %H:%M:%S")
sparql_log_fp = os.path.join(get_fp('data_dir/sparql_logs'), f"qald_io_util_{timestamp}.txt")
create_directory_if_not_exists(sparql_log_fp)
misc_consts.sparql_log_filehandle = open(sparql_log_fp, 'a', buffering=1) # buffering=1 for line-buffering

In [ ]:
from src.const.dataset import KgqaDataset, DatasetSplit
qald_dict = {
    # Train dataset not needed - ent-rel linkers might have bias
    'qald9plus_train': {
        'file_path': 'data_dir/processed_kgqa_ds/qald9plus/train/qald_9_train_linked.json',
        'name': 'QALD-9-plus - Train',
        'kgqa_ds': KgqaDataset.QALD9PLUS_UPDATED_TENTRISMAIN,
        'split': DatasetSplit.TRAIN
    },
    'qald9plus_test': {
        'file_path': 'data_dir/processed_kgqa_ds/qald9plus/test/qald_9_augmented_final.json',
        'name': 'QALD-9-plus - Test',
        'kgqa_ds': KgqaDataset.QALD9PLUS_UPDATED_TENTRISMAIN,
        'split': DatasetSplit.TEST
    },
    'qald10_test': {
        'file_path': 'data_dir/processed_kgqa_ds/qald10/test/qald_10_augmented_final.json',
        'name': 'QALD-10 - Test',
        'kgqa_ds': KgqaDataset.QALD10_UPDATED_TENTRISMAIN,
        'split': DatasetSplit.TEST,
        # Obsolete: This has been fixed: 'ignore_ids': [92, 203] # qald10 question that crashes tentris endpoint (better to avoid queries with sum)
    }
}

In [ ]:
## Obsolete -- Update augmented file to encapsulate ent-rel links
# from src.util.qald_io import encapsulate_qald_aug_info

# entrel_linker_name = 'aug_linker_v0.1'

# for qald_ds in qald_dict:
#     print(f'Processing: {qald_ds}')
#     encapsulate_qald_aug_info(qald_dict[qald_ds]['name'], entrel_linker_name, get_fp(qald_dict[qald_ds]['file_path']))


In [ ]:
## Append Gold entities and relations
from src.util.qald_io import update_qald_gold_info
from src.const.misc import WIKIDATA_PROP_INFO_CACHE_FILEPATH
for qald_ds in qald_dict:
    print(f'Processing: {qald_ds}')
    ds_dict = qald_dict[qald_ds]
    kgqa_ds_obj = ds_dict['kgqa_ds'].value
    update_qald_gold_info(get_fp(ds_dict['file_path']), kgqa_ds_obj.preferred_wd_endpoint, get_fp(WIKIDATA_PROP_INFO_CACHE_FILEPATH))

In [ ]:
## Update QALD dataset with latest answers from the given endpoint
from src.util.qald_io import update_qald_answers

for qald_ds in qald_dict:
    
    cur_ds_dict = qald_dict[qald_ds]
    
    print(f'Processing: {qald_ds}')
    cur_ds = cur_ds_dict['kgqa_ds'].value

    split_conf = cur_ds_dict['split']
    qald_file_path = get_fp(cur_ds_dict['file_path'])

    output_file_path = get_fp(f"{cur_ds.split_dict[split_conf]}")

    ignore_ids = cur_ds_dict['ignore_ids'] if 'ignore_ids' in cur_ds_dict else []

    failed_updates, ignored_updates = update_qald_answers(qald_file_path, output_file_path, cur_ds.preferred_wd_endpoint, ignore_ids)
    # Save failed updates somewhere
    print(f'Total {len(failed_updates)} question sparqls failed to fetch results.')
    print("Failed Items:")
    for item in failed_updates:
        print(f'{item['question'][0]['string']}\t{item['query']['sparql']}')

    print("Ignored Items:")
    for item in ignored_updates:
        print(f'{item['question'][0]['string']}\t{item['query']['sparql']}')
    print('\n\n')

In [ ]:
## Obsolete - Enrich QALD9Plus with QALD9 multilingual questions
# from src.util.qald_io import fetch_qald9_multilingual_strings
# qald9_dir = 'data_dir/kgqa_datasets/qald9'
# qald9plus_test = 'data_dir/processed_kgqa_ds/qald9plus/test/tentrisq10_aug_gold.json'
# #qald9plus_train = 'data_dir/processed_kgqa_ds/qald9plus/train/tentrisq10_aug_gold.json'

# fetch_qald9_multilingual_strings(get_fp(qald9_dir), get_fp(qald9plus_test))# , get_fp(qald9plus_train))

In [ ]:
## Clean QALD files for Gerbil
from src.util.qald_io import clean_qald_gerbil_json
for qald_ds in qald_dict:
    cur_ds_dict = qald_dict[qald_ds]
    
    print(f'Processing: {qald_ds}')
    cur_ds = cur_ds_dict['kgqa_ds'].value
    split_conf = cur_ds_dict['split']
    qald_file_path = get_fp(cur_ds.split_dict[split_conf])
    
    clean_qald_gerbil_json(qald_file_path)

In [ ]:
## Convert LcQUAD2 file to QALD format
from src.util.qald_io import convert_lcquad2_to_qald
from src.const.dataset import KgqaDataset, DatasetSplit

lcq2_ds = KgqaDataset.LCQUAD2_UPDATED_TENTRISMAIN.value
cur_split = DatasetSplit.TEST

lcq2_inp_file_path = get_fp('data_dir/processed_kgqa_ds/lcquad2/test/lcquad_aug_t5_corr.json')
lcq2_out_file_path = get_fp(lcq2_ds.split_dict[cur_split])

failed_update_items, missing_question_items = convert_lcquad2_to_qald(lcq2_inp_file_path, lcq2_out_file_path, lcq2_ds.preferred_wd_endpoint, lcq2_ds.use_sleep)
# Save failed updates somewhere
print(f'Total {len(failed_update_items)} question sparqls failed to fetch results.')
print("Failed Items:")
for item in failed_update_items:
    print(f'{item['question']}\t{item['sparql_wikidata']}')

print(f"Missing {len(missing_question_items)} Question or Augmented Text:")
for item in missing_question_items:
    print(f'{item['uid']}\t{item['sparql_wikidata']}')
print('\n\n')

In [ ]:
## Convert output tsv of Basic Factoid Solver to QALD format
from src.util.qald_io import convert_basic_output

tsv_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/tsv/aug_pred_gt.tsv")
output_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/json/aug_pred_gt.json")
has_tuples = True

qald_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/aug_gold.json")

convert_basic_output(tsv_file_path, qald_file_path, output_file_path, has_tuples=has_tuples)

In [ ]:
## Convert output tsv of Basic SPARQL Generator to QALD format
from src.util.qald_io import convert_basic_output

# tsv_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/tsv/aug_pred_sparql.tsv")
# output_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/json/aug_pred_sparql.json")

# tsv_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/tsv/baseline_pred_sparql.tsv")
# output_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/prediction/json/baseline_pred_sparql.json")

tsv_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald10/test/prediction/tsv/aug_pred_sparql.tsv")
output_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald10/test/prediction/json/aug_pred_sparql.json")

has_tuples = False

# qald_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald9plus/test/aug_gold.json")
qald_file_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/processed_kgqa_ds/qald10/test/aug_gold.json")

convert_basic_output(tsv_file_path, qald_file_path, output_file_path, has_tuples=has_tuples)

In [ ]:
## Convert SPINACH dataset to QALD-format
from src.const.dataset import KgqaDataset, DatasetSplit
from src.util.qald_io import convert_spinach_to_qald

spinach_ds = KgqaDataset.SPINACH_CURWD.value

wd_ep = spinach_ds.preferred_wd_endpoint

# split_conf = DatasetSplit.TEST
# input_path = get_fp('data_dir/processed_kgqa_ds/spinach/test/test.json')

split_conf = DatasetSplit.TRAIN
input_path = get_fp('data_dir/processed_kgqa_ds/spinach/train/dev.json')

out_dir = os.path.dirname(input_path)
out_file_name = 'qald_' + os.path.basename(input_path)

output_file_path = os.path.join(out_dir, out_file_name)

dataset_name = f'{spinach_ds.dataset_name} - {split_conf.name}'

convert_spinach_to_qald(dataset_name, input_path, output_file_path, wd_ep, use_sleep=True)

In [ ]:
## Format annotated SPINACH for experiments
from src.const.dataset import KgqaDataset, DatasetSplit
from src.util.qald_io import reformat_spinach_qald, update_qald_gold_info, clean_qald_gerbil_json
from src.const.misc import WIKIDATA_PROP_INFO_CACHE_FILEPATH
spinach_ds = KgqaDataset.SPINACH_CURWD.value
test_qald_path = spinach_ds.split_dict[DatasetSplit.TEST]
# Fix the formatting inconsistency
reformat_spinach_qald(get_fp(test_qald_path))
# Append Gold Entities and Relations (use_sleep=True)
update_qald_gold_info(get_fp(test_qald_path), spinach_ds.preferred_wd_endpoint, get_fp(WIKIDATA_PROP_INFO_CACHE_FILEPATH), spinach_ds.use_sleep)
# Clean QALD files for GERBIL
clean_qald_gerbil_json(get_fp(test_qald_path))

In [ ]:
# NOTE: Does not work because SPINACH uses federated SPARQLs
## Update SPINACH on local Tentris

# from src.const.dataset import KgqaDataset, DatasetSplit
# from src.util.qald_io import update_qald_answers, update_qald_gold_info, clean_qald_gerbil_json
# from src.const.misc import WIKIDATA_PROP_INFO_CACHE_FILEPATH

# input_qald_path = 'data_dir/processed_kgqa_ds/spinach/test/qald_test_final.json'

# spinach_ds = KgqaDataset.SPINACH_TENTRIS_WIKI.value
# test_qald_path = spinach_ds.split_dict[DatasetSplit.TEST]
# # Update spinach answers
# failed_updates, ignored_updates = update_qald_answers(get_fp(input_qald_path), get_fp(test_qald_path), spinach_ds.preferred_wd_endpoint, [])
# # Save failed updates somewhere
# print(f'Total {len(failed_updates)} question sparqls failed to fetch results.')
# print("Failed Items:")
# for item in failed_updates:
#     print(f'{item['question'][0]['string']}\t{item['query']['sparql']}')

# print("Ignored Items:")
# for item in ignored_updates:
#     print(f'{item['question'][0]['string']}\t{item['query']['sparql']}')
# print('\n\n')

# # Append Gold Entities and Relations (use_sleep=True)
# update_qald_gold_info(get_fp(test_qald_path), spinach_ds.preferred_wd_endpoint, get_fp(WIKIDATA_PROP_INFO_CACHE_FILEPATH), spinach_ds.use_sleep)
# # Clean QALD files for GERBIL
# clean_qald_gerbil_json(get_fp(test_qald_path))

In [ ]:
misc_consts.sparql_log_filehandle.close()